# V3-1 CatBoost Colab 리뷰 노트북

이 노트북은 **V3-1을 처음부터 끝까지 다시 학습**해서 `submission.csv`를 만드는 리뷰용 실행본입니다.

팀이 셀 단위로 읽고 리뷰하기 쉽도록 파트를 나눠 두었습니다.

요약
- 입력: `train.csv`, `test.csv`, `sample_submission.csv`
- 출력: `submission.csv`, `oof_predictions.csv`, `feature_importance.csv`, `fold_scores.csv`, `run_summary.json`
- 학습: `5-fold x 3-seed`, CatBoost, IVF/DI 분기 피처 포함
- 편의 기능: Colab에서 CSV 직접 업로드 가능, 학습 후 결과 파일 자동 다운로드


## 실행 순서

1. 패키지 설치 셀을 실행합니다.
2. 필요하면 CSV 직접 업로드 셀을 실행합니다.
3. `DATA_DIR`, `OUTPUT_DIR` 설정을 확인합니다.
4. 아래 파트 셀을 위에서 아래로 순서대로 실행합니다.
5. dry-run 계획을 먼저 확인한 뒤 실제 학습 셀을 실행합니다.


In [ ]:
# Part 0. Colab 패키지 설치
!pip -q install numpy pandas scikit-learn catboost


In [ ]:
# Part 0-1. CSV 직접 업로드 (Colab 전용)
from pathlib import Path

UPLOADED_DATA_DIR = Path("/content/v3_1_uploaded_data")
USE_UPLOADED_FILES = False

try:
    from google.colab import files  # type: ignore
except ModuleNotFoundError:
    print("Colab 환경이 아니어서 업로드 셀은 건너뜁니다.")
else:
    print("train.csv, test.csv, sample_submission.csv를 직접 업로드할 수 있습니다.")
    uploaded = files.upload()
    if uploaded:
        UPLOADED_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for filename, content in uploaded.items():
            (UPLOADED_DATA_DIR / filename).write_bytes(content)
        USE_UPLOADED_FILES = True
        print({"업로드_경로": str(UPLOADED_DATA_DIR), "업로드_파일": sorted(uploaded.keys())})
    else:
        print("업로드한 파일이 없으면 아래 DATA_DIR 설정값을 그대로 사용합니다.")


In [ ]:
# Part 0-2. 데이터/출력 경로 설정
# 구글 드라이브에 데이터가 있으면 아래 예시처럼 연결해서 DATA_DIR를 바꿀 수 있습니다.
# from google.colab import drive
# drive.mount("/content/drive")
# DEFAULT_DATA_DIR = Path("/content/drive/MyDrive/open")

from pathlib import Path

DEFAULT_DATA_DIR = Path("/content/data")
DATA_DIR = UPLOADED_DATA_DIR if globals().get("USE_UPLOADED_FILES", False) else DEFAULT_DATA_DIR
OUTPUT_DIR = Path("/content/v3_1_outputs")
TASK_TYPE = "AUTO"  # AUTO / CPU / GPU
DEVICES = "0"
AUTO_INSTALL_MISSING = True

print({
    "DATA_DIR": str(DATA_DIR),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "TASK_TYPE": TASK_TYPE,
    "직접업로드사용": bool(globals().get("USE_UPLOADED_FILES", False)),
})


## Part 1. 공통 상수와 설정

- 원본 CSV 컬럼명과 매핑
- 나이/횟수 변환 규칙
- 학습 설정 dataclass


In [ ]:
# Part 1. 공통 상수와 설정
import importlib.util
import json
import subprocess
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
SUBMISSION_FILE = "sample_submission.csv"
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "catboost": "catboost",
}

ID_COLUMN = "ID"
TARGET_COLUMN = "임신 성공 여부"

AGE_COLUMN = "시술 당시 나이"
TREATMENT_COLUMN = "시술 유형"
SPECIFIC_TREATMENT_COLUMN = "특정 시술 유형"
TIME_CODE_COLUMN = "시술 시기 코드"
EMBRYO_REASON_COLUMN = "배아 생성 주요 이유"
OOCYTE_SOURCE_COLUMN = "난자 출처"
SPERM_SOURCE_COLUMN = "정자 출처"

COUNT_TEXT_COLUMNS = {
    "총 시술 횟수": "total_treatment_count_ord",
    "클리닉 내 총 시술 횟수": "clinic_treatment_count_ord",
    "IVF 시술 횟수": "ivf_treatment_count_ord",
    "DI 시술 횟수": "di_treatment_count_ord",
    "총 임신 횟수": "total_pregnancy_count_ord",
    "IVF 임신 횟수": "ivf_pregnancy_count_ord",
    "DI 임신 횟수": "di_pregnancy_count_ord",
    "총 출산 횟수": "total_birth_count_ord",
    "IVF 출산 횟수": "ivf_birth_count_ord",
    "DI 출산 횟수": "di_birth_count_ord",
}

AGE_MAPPING = {
    "만18-34세": 0,
    "만35-37세": 1,
    "만38-39세": 2,
    "만40-42세": 3,
    "만43-44세": 4,
    "만45-50세": 5,
    "알 수 없음": 6,
}

COUNT_MAPPING = {
    "0회": 0,
    "1회": 1,
    "2회": 2,
    "3회": 3,
    "4회": 4,
    "5회": 5,
    "6회 이상": 6,
}

INFERTILITY_CAUSE_COLUMNS = (
    "불임 원인 - 난관 질환",
    "불임 원인 - 남성 요인",
    "불임 원인 - 배란 장애",
    "불임 원인 - 여성 요인",
    "불임 원인 - 자궁경부 문제",
    "불임 원인 - 자궁내막증",
    "불임 원인 - 정자 농도",
    "불임 원인 - 정자 면역학적 요인",
    "불임 원인 - 정자 운동성",
    "불임 원인 - 정자 형태",
)

HIGH_MISSING_COLUMNS = (
    "난자 해동 경과일",
    "PGS 시술 여부",
    "PGD 시술 여부",
    "착상 전 유전 검사 사용 여부",
    "임신 시도 또는 마지막 임신 경과 연수",
    "배아 해동 경과일",
    "난자 채취 경과일",
    "난자 혼합 경과일",
    "배아 이식 경과일",
)

NUMERIC_SOURCE_COLUMNS = (
    "총 생성 배아 수",
    "미세주입된 난자 수",
    "미세주입에서 생성된 배아 수",
    "이식된 배아 수",
    "미세주입 배아 이식 수",
    "저장된 배아 수",
    "미세주입 후 저장된 배아 수",
    "해동된 배아 수",
    "해동 난자 수",
    "수집된 신선 난자 수",
    "저장된 신선 난자 수",
    "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수",
    "기증자 정자와 혼합된 난자 수",
    "난자 채취 경과일",
    "난자 해동 경과일",
    "난자 혼합 경과일",
    "배아 이식 경과일",
    "배아 해동 경과일",
    "임신 시도 또는 마지막 임신 경과 연수",
)

EMBRYO_USAGE_COLUMNS = (
    "동결 배아 사용 여부",
    "신선 배아 사용 여부",
    "기증 배아 사용 여부",
)

COLAB_DATA_CANDIDATES = (
    Path("/content/open"),
    Path("/content/data"),
    Path("/content/drive/MyDrive/open"),
    Path("/content/drive/MyDrive/data"),
    Path("/content/drive/MyDrive/dacon/open"),
    Path("/content/drive/MyDrive/dacon/data"),
)


@dataclass(frozen=True)
class FeatureReport:
    feature_version: str
    base_feature_version: str
    added_numeric: tuple[str, ...]
    added_categorical: tuple[str, ...]
    missing_flags: tuple[str, ...]


@dataclass(frozen=True)
class V31Config:
    data_dir: Path
    output_dir: Path
    experiment_name: str = "catboost_fe_v3_1_ivf_di_split"
    id_column: str = ID_COLUMN
    target_column: str = TARGET_COLUMN
    train_file: str = TRAIN_FILE
    test_file: str = TEST_FILE
    submission_file: str = SUBMISSION_FILE
    folds: int = 5
    seeds: tuple[int, ...] = (42, 2026, 2604)
    iterations: int = 6000
    learning_rate: float = 0.02
    depth: int = 6
    l2_leaf_reg: float = 10.0
    random_strength: float = 1.2
    bagging_temperature: float = 0.8
    rsm: float = 0.8
    border_count: int = 128
    early_stopping_rounds: int = 300
    task_type: str = "AUTO"
    devices: str | None = "0"
    auto_install_missing: bool = True


## Part 2. 공통 베이스 피처 엔지니어링

- `lgbm_fe_v1` 기반 공통 파생 피처를 만듭니다.
- 결측 플래그, ordinal 매핑, 비율/차이, 조합형 categorical이 여기서 생성됩니다.


In [ ]:
# Part 2. 공통 베이스 피처 엔지니어링
def _to_numeric(series):
    import pandas as pd

    return pd.to_numeric(series, errors="coerce")


def _safe_ratio(numerator, denominator):
    import numpy as np

    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def _as_category(series):
    return series.fillna("<MISSING>").astype(str)


def _safe_numeric(frame, column: str):
    import pandas as pd

    if column not in frame.columns:
        return pd.Series(0.0, index=frame.index, dtype="float32")
    return pd.to_numeric(frame[column], errors="coerce").fillna(0.0).astype("float32")


def _apply_common_transforms(df):
    frame = df.copy()

    for column in NUMERIC_SOURCE_COLUMNS:
        if column in frame.columns:
            frame[column] = _to_numeric(frame[column]).astype("float32")

    frame["age_bucket_ord"] = _as_category(frame[AGE_COLUMN]).map(AGE_MAPPING).astype("float32")

    for source_column, feature_name in COUNT_TEXT_COLUMNS.items():
        frame[feature_name] = _as_category(frame[source_column]).map(COUNT_MAPPING).astype("float32")

    for column in HIGH_MISSING_COLUMNS:
        if column in frame.columns:
            frame[f"{column}_missing_flag"] = frame[column].isna().astype("int8")

    frame["treatment_specific_combo"] = (
        _as_category(frame[TREATMENT_COLUMN]) + "__" + _as_category(frame[SPECIFIC_TREATMENT_COLUMN])
    )
    frame["age_treatment_combo"] = _as_category(frame[AGE_COLUMN]) + "__" + _as_category(frame[TREATMENT_COLUMN])
    frame["source_combo"] = _as_category(frame[OOCYTE_SOURCE_COLUMN]) + "__" + _as_category(frame[SPERM_SOURCE_COLUMN])
    frame["embryo_usage_combo"] = (
        _as_category(frame[EMBRYO_USAGE_COLUMNS[0]])
        + "__"
        + _as_category(frame[EMBRYO_USAGE_COLUMNS[1]])
        + "__"
        + _as_category(frame[EMBRYO_USAGE_COLUMNS[2]])
    )
    frame["treatment_time_combo"] = _as_category(frame[TREATMENT_COLUMN]) + "__" + _as_category(frame[TIME_CODE_COLUMN])
    frame["embryo_reason_treatment_combo"] = (
        _as_category(frame[EMBRYO_REASON_COLUMN]) + "__" + _as_category(frame[TREATMENT_COLUMN])
    )

    cause_values = frame.loc[:, INFERTILITY_CAUSE_COLUMNS].fillna(0).astype("float32")
    frame["infertility_cause_count"] = cause_values.sum(axis=1).astype("float32")
    frame["major_infertility_bundle"] = (
        _as_category(frame["불임 원인 - 남성 요인"])
        + "__"
        + _as_category(frame["불임 원인 - 배란 장애"])
        + "__"
        + _as_category(frame["불임 원인 - 난관 질환"])
    )

    total_created = frame["총 생성 배아 수"]
    transferred = frame["이식된 배아 수"]
    stored = frame["저장된 배아 수"]
    injected_oocytes = frame["미세주입된 난자 수"]
    injected_embryos = frame["미세주입에서 생성된 배아 수"]
    mixed_oocytes = frame["혼합된 난자 수"]
    partner_mixed = frame["파트너 정자와 혼합된 난자 수"]
    donor_mixed = frame["기증자 정자와 혼합된 난자 수"]
    fresh_oocytes = frame["수집된 신선 난자 수"]
    embryo_transfer_days = frame["배아 이식 경과일"]
    pregnancy_gap_years = frame["임신 시도 또는 마지막 임신 경과 연수"]

    frame["transfer_over_created_ratio"] = _safe_ratio(transferred, total_created).astype("float32")
    frame["stored_over_created_ratio"] = _safe_ratio(stored, total_created).astype("float32")
    frame["created_minus_transferred"] = (total_created - transferred).astype("float32")
    frame["created_minus_stored"] = (total_created - stored).astype("float32")
    frame["injected_embryo_ratio"] = _safe_ratio(injected_embryos, injected_oocytes).astype("float32")
    frame["partner_mixed_ratio"] = _safe_ratio(partner_mixed, mixed_oocytes).astype("float32")
    frame["donor_mixed_ratio"] = _safe_ratio(donor_mixed, mixed_oocytes).astype("float32")
    frame["mixed_minus_partner"] = (mixed_oocytes - partner_mixed).astype("float32")
    frame["fresh_minus_mixed"] = (fresh_oocytes - mixed_oocytes).astype("float32")
    frame["embryo_transfer_days_x_age"] = (embryo_transfer_days * frame["age_bucket_ord"]).astype("float32")
    frame["pregnancy_gap_x_age"] = (pregnancy_gap_years * frame["age_bucket_ord"]).astype("float32")
    frame["total_treatment_x_age"] = (frame["total_treatment_count_ord"] * frame["age_bucket_ord"]).astype("float32")

    return frame


## Part 3. V3-1 IVF/DI 분기 피처

- IVF와 DI를 같은 방식으로 보지 않도록 시술 유형별 전용 채널을 추가합니다.
- V3-1이 V2보다 좋아졌던 핵심 아이디어가 이 파트입니다.


In [ ]:
# Part 3. V3-1 IVF/DI 분기 피처
def _apply_ivf_di_split_features(train_frame, test_frame):
    numeric_features: list[str] = []
    categorical_features: list[str] = []

    for frame in (train_frame, test_frame):
        treatment = _as_category(frame[TREATMENT_COLUMN]).str.upper()
        is_ivf = (treatment == "IVF").astype("float32")
        is_di = (treatment == "DI").astype("float32")

        frame["is_ivf_flag_v3"] = is_ivf
        frame["is_di_flag_v3"] = is_di
        frame["ivf_total_treatment_load_v3"] = is_ivf * _safe_numeric(frame, "total_treatment_count_ord")
        frame["di_total_treatment_load_v3"] = is_di * _safe_numeric(frame, "total_treatment_count_ord")
        frame["ivf_pregnancy_load_v3"] = is_ivf * _safe_numeric(frame, "total_pregnancy_count_ord")
        frame["di_pregnancy_load_v3"] = is_di * _safe_numeric(frame, "total_pregnancy_count_ord")
        frame["ivf_transfer_ratio_signal_v3"] = is_ivf * _safe_numeric(frame, "transfer_over_created_ratio")
        frame["di_transfer_ratio_signal_v3"] = is_di * _safe_numeric(frame, "transfer_over_created_ratio")
        frame["ivf_storage_ratio_signal_v3"] = is_ivf * _safe_numeric(frame, "stored_over_created_ratio")
        frame["di_storage_ratio_signal_v3"] = is_di * _safe_numeric(frame, "stored_over_created_ratio")
        frame["ivf_embryo_age_signal_v3"] = is_ivf * _safe_numeric(frame, "embryo_transfer_days_x_age")
        frame["di_embryo_age_signal_v3"] = is_di * _safe_numeric(frame, "embryo_transfer_days_x_age")

        frame["treatment_history_combo_v3"] = treatment + "__" + _as_category(frame["total_treatment_count_ord"])
        frame["treatment_source_combo_v3"] = treatment + "__" + _as_category(frame["source_combo"])
        frame["treatment_age_history_combo_v3"] = (
            treatment + "__" + _as_category(frame[AGE_COLUMN]) + "__" + _as_category(frame["total_treatment_count_ord"])
        )

    numeric_features.extend(
        [
            "is_ivf_flag_v3",
            "is_di_flag_v3",
            "ivf_total_treatment_load_v3",
            "di_total_treatment_load_v3",
            "ivf_pregnancy_load_v3",
            "di_pregnancy_load_v3",
            "ivf_transfer_ratio_signal_v3",
            "di_transfer_ratio_signal_v3",
            "ivf_storage_ratio_signal_v3",
            "di_storage_ratio_signal_v3",
            "ivf_embryo_age_signal_v3",
            "di_embryo_age_signal_v3",
        ]
    )
    categorical_features.extend(
        [
            "treatment_history_combo_v3",
            "treatment_source_combo_v3",
            "treatment_age_history_combo_v3",
        ]
    )
    return train_frame, test_frame, numeric_features, categorical_features


def _align_categories(train_df, test_df):
    import pandas as pd

    train_frame = train_df.copy()
    test_frame = test_df.copy()

    categorical_columns: list[str] = []
    for column in train_frame.columns:
        if column in {ID_COLUMN, TARGET_COLUMN}:
            continue
        if str(train_frame[column].dtype) in {"object", "category", "string", "str"}:
            categorical_columns.append(column)

    for column in categorical_columns:
        combined = pd.concat(
            [_as_category(train_frame[column]), _as_category(test_frame[column])],
            axis=0,
        )
        categories = pd.Index(combined.unique())
        train_frame[column] = pd.Categorical(_as_category(train_frame[column]), categories=categories)
        test_frame[column] = pd.Categorical(_as_category(test_frame[column]), categories=categories)

    return train_frame, test_frame, categorical_columns


def prepare_v3_1_frames(train_df, test_df):
    train_frame = _apply_common_transforms(train_df)
    test_frame = _apply_common_transforms(test_df)
    train_frame, test_frame, added_numeric, added_categorical = _apply_ivf_di_split_features(
        train_frame,
        test_frame,
    )
    train_frame, test_frame, categorical_columns = _align_categories(train_frame, test_frame)

    feature_columns = [column for column in train_frame.columns if column not in {ID_COLUMN, TARGET_COLUMN}]
    missing_flags = tuple(
        f"{column}_missing_flag"
        for column in HIGH_MISSING_COLUMNS
        if column in train_df.columns
    )
    report = FeatureReport(
        feature_version="catboost_enable_ivf_di_split",
        base_feature_version="lgbm_fe_v1",
        added_numeric=tuple(added_numeric),
        added_categorical=tuple(added_categorical),
        missing_flags=missing_flags,
    )
    return train_frame, test_frame, feature_columns, categorical_columns, report


## Part 4. 런타임 유틸리티

- Colab 환경 감지
- 필요 패키지 자동 설치
- 데이터 경로 자동 탐색
- dry-run 실행 계획 출력


In [ ]:
# Part 4. 런타임 유틸리티
def in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True


def dependency_status() -> dict[str, bool]:
    return {import_name: importlib.util.find_spec(import_name) is not None for import_name in REQUIRED_PACKAGES}


def install_missing_packages() -> list[str]:
    missing = [
        pip_name
        for import_name, pip_name in REQUIRED_PACKAGES.items()
        if importlib.util.find_spec(import_name) is None
    ]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    return missing


def detect_default_data_dir(script_path: Path) -> Path:
    if in_colab():
        for candidate in COLAB_DATA_CANDIDATES:
            if all((candidate / name).exists() for name in (TRAIN_FILE, TEST_FILE, SUBMISSION_FILE)):
                return candidate

    package_dir = script_path.resolve().parent
    candidates: list[Path] = []
    for parent in [package_dir, *package_dir.parents]:
        candidates.append(parent / "open")
        candidates.append(parent / "data")

    for candidate in candidates:
        if all((candidate / name).exists() for name in (TRAIN_FILE, TEST_FILE, SUBMISSION_FILE)):
            return candidate

    return Path("/content/data") if in_colab() else package_dir / "data"


def required_data_files(config: V31Config) -> tuple[Path, ...]:
    return (
        config.data_dir / config.train_file,
        config.data_dir / config.test_file,
        config.data_dir / config.submission_file,
    )


def validate_data_layout(config: V31Config) -> list[str]:
    return [path.name for path in required_data_files(config) if not path.exists()]


def default_config(script_path: Path, data_dir: Path | None = None, output_dir: Path | None = None) -> V31Config:
    resolved_data_dir = data_dir or detect_default_data_dir(script_path)
    if output_dir is not None:
        resolved_output_dir = output_dir
    elif in_colab():
        resolved_output_dir = Path("/content/v3_1_outputs")
    else:
        resolved_output_dir = script_path.resolve().parent / "outputs" / "v3_1_colab"
    return V31Config(data_dir=resolved_data_dir, output_dir=resolved_output_dir)


def build_execution_plan(config: V31Config) -> dict[str, object]:
    return {
        "runtime": {
            "in_colab": in_colab(),
            "auto_install_missing": config.auto_install_missing,
            "colab_candidates": [str(path) for path in COLAB_DATA_CANDIDATES],
        },
        "pipeline": [
            "install missing packages when needed",
            "read train.csv / test.csv / sample_submission.csv",
            "apply base lgbm_fe_v1 feature engineering",
            "add IVF/DI split V3-1 features",
            "align categorical spaces",
            "train CatBoost with 5-fold x 3-seed",
            "save submission, OOF, importance, fold scores, summary",
        ],
        "data_dir": str(config.data_dir),
        "output_dir": str(config.output_dir),
        "dependency_status": dependency_status(),
        "missing_files": validate_data_layout(config),
        "config": {
            "experiment_name": config.experiment_name,
            "folds": config.folds,
            "seeds": config.seeds,
            "iterations": config.iterations,
            "learning_rate": config.learning_rate,
            "depth": config.depth,
            "l2_leaf_reg": config.l2_leaf_reg,
            "random_strength": config.random_strength,
            "bagging_temperature": config.bagging_temperature,
            "rsm": config.rsm,
            "border_count": config.border_count,
            "early_stopping_rounds": config.early_stopping_rounds,
            "task_type": config.task_type,
            "devices": config.devices,
        },
    }


def import_training_stack():
    try:
        import numpy as np
        import pandas as pd
        from catboost import CatBoostClassifier, Pool
        from catboost.utils import get_gpu_device_count
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import StratifiedKFold
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "이 스크립트는 numpy, pandas, scikit-learn, catboost 설치가 필요합니다."
        ) from exc
    return np, pd, CatBoostClassifier, Pool, get_gpu_device_count, roc_auc_score, StratifiedKFold


def ensure_training_stack(config: V31Config):
    if config.auto_install_missing:
        install_missing_packages()
    return import_training_stack()


def resolve_task_type(config: V31Config) -> tuple[str, str | None]:
    requested = config.task_type.upper()
    if requested in {"GPU", "CPU"}:
        return requested, config.devices

    _, _, _, _, get_gpu_device_count, _, _ = import_training_stack()
    try:
        gpu_count = get_gpu_device_count()
    except Exception:
        gpu_count = 0

    if gpu_count and gpu_count > 0:
        return "GPU", config.devices
    return "CPU", None


def _prepare_catboost_frames(train_frame, test_frame, feature_columns: list[str], categorical_columns: list[str]):
    prepared_train = train_frame.copy()
    prepared_test = test_frame.copy()
    numeric_columns = [column for column in feature_columns if column not in categorical_columns]

    for column in categorical_columns:
        prepared_train[column] = (
            prepared_train[column].astype("object").where(prepared_train[column].notna(), "<MISSING>").astype(str)
        )
        prepared_test[column] = (
            prepared_test[column].astype("object").where(prepared_test[column].notna(), "<MISSING>").astype(str)
        )

    for column in numeric_columns:
        prepared_train[column] = prepared_train[column]
        prepared_test[column] = prepared_test[column]

    return prepared_train, prepared_test


def write_json(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def resolve_submission_column(submission_df, id_column: str) -> str:
    if "probability" in submission_df.columns:
        return "probability"

    candidate_columns = [column for column in submission_df.columns if column != id_column]
    if len(candidate_columns) == 1:
        return candidate_columns[0]

    raise RuntimeError("sample_submission.csv에서 예측값 컬럼을 찾지 못했습니다.")


## Part 5. CatBoost 학습 루프

- 5-fold x 3-seed로 학습합니다.
- OOF, feature importance, fold score, submission을 저장합니다.


In [ ]:
# Part 5. CatBoost 학습 루프
def run_training(config: V31Config, dry_run: bool = False) -> int:
    plan = build_execution_plan(config)
    if dry_run:
        print(json.dumps(plan, ensure_ascii=False, indent=2))
        return 0

    missing_files = validate_data_layout(config)
    if missing_files:
        raise RuntimeError(f"데이터 파일이 없습니다: {', '.join(missing_files)}")

    np, pd, CatBoostClassifier, Pool, _, roc_auc_score, StratifiedKFold = ensure_training_stack(config)

    task_type, devices = resolve_task_type(config)
    config = V31Config(**{**asdict(config), "task_type": task_type, "devices": devices})

    train_df = pd.read_csv(config.data_dir / config.train_file, encoding="utf-8-sig")
    test_df = pd.read_csv(config.data_dir / config.test_file, encoding="utf-8-sig")
    submission_df = pd.read_csv(config.data_dir / config.submission_file, encoding="utf-8-sig")

    train_frame, test_frame, feature_columns, categorical_columns, feature_report = prepare_v3_1_frames(
        train_df,
        test_df,
    )
    train_frame, test_frame = _prepare_catboost_frames(
        train_frame,
        test_frame,
        feature_columns,
        categorical_columns,
    )

    x_train = train_frame.loc[:, feature_columns]
    y_train = train_frame[config.target_column].astype(int)
    x_test = test_frame.loc[:, feature_columns]

    oof_predictions = np.zeros(len(x_train), dtype=float)
    test_predictions = np.zeros(len(x_test), dtype=float)
    fold_scores: list[dict[str, object]] = []
    seed_scores: list[dict[str, object]] = []
    importance_frames: list = []

    for seed in config.seeds:
        splitter = StratifiedKFold(
            n_splits=config.folds,
            shuffle=True,
            random_state=seed,
        )
        seed_oof = np.zeros(len(x_train), dtype=float)
        seed_test = np.zeros(len(x_test), dtype=float)

        for fold, (train_index, valid_index) in enumerate(splitter.split(x_train, y_train), start=1):
            x_fold_train = x_train.iloc[train_index].copy()
            y_fold_train = y_train.iloc[train_index].copy()
            x_fold_valid = x_train.iloc[valid_index].copy()
            y_fold_valid = y_train.iloc[valid_index].copy()

            train_pool = Pool(x_fold_train, y_fold_train, cat_features=categorical_columns)
            valid_pool = Pool(x_fold_valid, y_fold_valid, cat_features=categorical_columns)
            test_pool = Pool(x_test, cat_features=categorical_columns)

            model_kwargs = {
                "loss_function": "Logloss",
                "eval_metric": "AUC",
                "iterations": config.iterations,
                "learning_rate": config.learning_rate,
                "depth": config.depth,
                "l2_leaf_reg": config.l2_leaf_reg,
                "random_strength": config.random_strength,
                "bagging_temperature": config.bagging_temperature,
                "border_count": config.border_count,
                "early_stopping_rounds": config.early_stopping_rounds,
                "auto_class_weights": "Balanced",
                "random_seed": seed,
                "allow_writing_files": False,
                "verbose": False,
            }
            if config.task_type == "CPU":
                model_kwargs["thread_count"] = -1
                model_kwargs["rsm"] = config.rsm
            else:
                model_kwargs["task_type"] = config.task_type
                if config.devices:
                    model_kwargs["devices"] = config.devices

            model = CatBoostClassifier(**model_kwargs)
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

            valid_pred = model.predict_proba(valid_pool)[:, 1]
            test_pred = model.predict_proba(test_pool)[:, 1]

            seed_oof[valid_index] = valid_pred
            seed_test += test_pred / config.folds

            fold_auc = float(roc_auc_score(y_fold_valid, valid_pred))
            best_iteration = model.get_best_iteration()
            fold_scores.append(
                {
                    "seed": seed,
                    "fold": fold,
                    "roc_auc": round(fold_auc, 6),
                    "best_iteration": int(best_iteration) if best_iteration is not None else None,
                }
            )

            feature_importance = model.get_feature_importance(train_pool, type="FeatureImportance")
            importance_frames.append(
                pd.DataFrame(
                    {
                        "feature": feature_columns,
                        "importance_gain": feature_importance,
                        "importance_split": feature_importance,
                        "seed": seed,
                        "fold": fold,
                    }
                )
            )

        seed_auc = float(roc_auc_score(y_train, seed_oof))
        seed_scores.append({"seed": seed, "roc_auc": round(seed_auc, 6)})
        oof_predictions += seed_oof / len(config.seeds)
        test_predictions += seed_test / len(config.seeds)

    overall_auc = float(roc_auc_score(y_train, oof_predictions))
    importance_frame = (
        pd.concat(importance_frames, ignore_index=True)
        .groupby("feature", as_index=False)[["importance_gain", "importance_split"]]
        .mean()
        .sort_values("importance_gain", ascending=False)
    )

    config.output_dir.mkdir(parents=True, exist_ok=True)
    prediction_column = resolve_submission_column(submission_df, config.id_column)
    submission_df[prediction_column] = test_predictions
    submission_df.to_csv(config.output_dir / "submission.csv", index=False, encoding="utf-8-sig")

    pd.DataFrame(
        {
            config.id_column: train_frame[config.id_column],
            "oof_probability": oof_predictions,
            config.target_column: y_train,
        }
    ).to_csv(config.output_dir / "oof_predictions.csv", index=False, encoding="utf-8-sig")
    importance_frame.to_csv(config.output_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(fold_scores).to_csv(config.output_dir / "fold_scores.csv", index=False, encoding="utf-8-sig")

    seed_frame = pd.DataFrame(seed_scores)
    summary = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "experiment_name": config.experiment_name,
        "overall_auc": round(overall_auc, 6),
        "seed_auc_mean": round(float(seed_frame["roc_auc"].mean()), 6),
        "seed_auc_std": round(float(seed_frame["roc_auc"].std(ddof=0)), 6),
        "feature_count": len(feature_columns),
        "categorical_count": len(categorical_columns),
        "feature_report": asdict(feature_report),
        "params": {
            "folds": config.folds,
            "seeds": config.seeds,
            "iterations": config.iterations,
            "learning_rate": config.learning_rate,
            "depth": config.depth,
            "l2_leaf_reg": config.l2_leaf_reg,
            "random_strength": config.random_strength,
            "bagging_temperature": config.bagging_temperature,
            "rsm": config.rsm,
            "border_count": config.border_count,
            "early_stopping_rounds": config.early_stopping_rounds,
            "task_type": config.task_type,
            "devices": config.devices,
        },
        "seed_scores": seed_scores,
        "fold_scores": fold_scores,
        "top_features": importance_frame.head(20).to_dict(orient="records"),
        "output_files": {
            "submission": str(config.output_dir / "submission.csv"),
            "oof": str(config.output_dir / "oof_predictions.csv"),
            "importance": str(config.output_dir / "feature_importance.csv"),
            "fold_scores": str(config.output_dir / "fold_scores.csv"),
        },
    }
    write_json(config.output_dir / "run_summary.json", summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return 0


## Part 6. Dry-run 계획 확인

- 실제 학습 전에 입력 경로, 출력 경로, 누락 파일 여부를 먼저 확인합니다.


In [ ]:
# Part 6. Dry-run 계획 확인
config = V31Config(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    task_type=TASK_TYPE,
    devices=DEVICES,
    auto_install_missing=AUTO_INSTALL_MISSING,
)

print(json.dumps(build_execution_plan(config), ensure_ascii=False, indent=2))


## Part 7. 실제 학습 실행

- 이 셀에서 실제 5-fold x 3-seed 학습이 시작됩니다.
- 학습이 끝나면 `submission.csv`를 자동 다운로드하고, 원본 `sample_submission.csv`도 함께 내려받을 수 있게 합니다.


In [ ]:
# Part 7. 실제 학습 실행
config = V31Config(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    task_type=TASK_TYPE,
    devices=DEVICES,
    auto_install_missing=AUTO_INSTALL_MISSING,
)

run_training(config, dry_run=False)

# 학습이 끝나면 결과 파일을 바로 내려받을 수 있게 합니다.
download_targets = [OUTPUT_DIR / "submission.csv"]
sample_submission_path = DATA_DIR / "sample_submission.csv"
if sample_submission_path.exists():
    download_targets.append(sample_submission_path)

for path in download_targets:
    print(f"다운로드 준비: {path}")

try:
    from google.colab import files  # type: ignore
except ModuleNotFoundError:
    print("Colab 환경이 아니어서 자동 다운로드는 건너뜁니다.")
else:
    for path in download_targets:
        if path.exists():
            files.download(str(path))
        else:
            print(f"파일이 없어 다운로드를 건너뜁니다: {path}")
